# 🚦 Sidra Intersection – Movement Summary Extractor

This notebook extracts key performance data from **SIDRA INTERSECTION** PDF reports and produces a clean Excel summary.

### Output Columns
| Column | Description |
|--------|-------------|
| Scenario | Time period / scenario name (e.g. Base AM, Base PM) |
| Location | Junction reference (e.g. Jn 1, Jn 2) |
| Overall Intersection Delay (sec) | Average delay for all vehicles at the junction |
| Overall LOS | Level of Service derived from overall delay |
| Worst Approach | The approach direction with the highest delay |
| Worst Approach Delay (sec) | Delay value for the worst approach |
| Worst Approach LOS | Level of Service for the worst approach |

### LOS Thresholds (HCM)
| LOS | Delay (sec) |
|-----|-------------|
| A | ≤ 10 |
| B | 10 – 15 |
| C | 15 – 25 |
| D | 25 – 35 |
| E | 35 – 50 |
| F | > 50 |

> **Note:** LOS is always calculated from delay values, so `NA` entries in the Sidra report are handled correctly.

---

## Step 1 – Install Required Libraries

Run this cell once. It installs:
- **pdfplumber** – extracts text and tables from PDFs
- **pymupdf** – splits multi-page PDFs into individual pages

In [ ]:
!pip install pdfplumber pymupdf --quiet

## Step 2 – Upload Sidra PDF Report(s)

A file picker will appear. Select **one or more** Sidra PDF reports to process.

> You can upload multiple PDFs at once (e.g. Base AM and Base PM reports).

In [ ]:
from google.colab import files

print("Please select your Sidra PDF file(s) to upload...")
uploaded = files.upload()

print(f"\n✅ {len(uploaded)} file(s) uploaded successfully:")
for name in uploaded.keys():
    print(f"   • {name}")

## Step 3 – Import Libraries & Define Helper Function

Imports all required libraries and defines the `delay_to_los()` function that converts average delay (in seconds) to an HCM Level of Service letter (A–F).

In [ ]:
import pandas as pd
import pdfplumber
import fitz          # pymupdf
import re
import os
import shutil
import warnings
import logging

# Suppress noisy warnings from PDF libraries
warnings.filterwarnings("ignore")
logging.getLogger("pdfminer").setLevel(logging.ERROR)


def delay_to_los(delay: float) -> str:
    """
    Convert average intersection delay (seconds) to HCM Level of Service.
    Always calculated from delay – never taken as 'NA' from the Sidra report.
    """
    if delay <= 10:  return "A"
    if delay <= 15:  return "B"
    if delay <= 25:  return "C"
    if delay <= 35:  return "D"
    if delay <= 50:  return "E"
    return "F"


print("✅ Libraries imported and helper function defined.")

## Step 4 – Split PDFs into Individual MOVEMENT SUMMARY Pages

Each Sidra PDF may contain multiple junctions. This step:
1. Scans every page of each uploaded PDF
2. Identifies pages that contain a **MOVEMENT SUMMARY** header
3. Saves each such page as a separate PDF in the `output_pages/` folder

This makes parsing each junction independently much more reliable.

In [ ]:
OUTPUT_FOLDER = "output_pages"

# Clean up any previous run
if os.path.exists(OUTPUT_FOLDER):
    shutil.rmtree(OUTPUT_FOLDER)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

count = 0
for filename in uploaded.keys():
    doc = fitz.open(filename)
    pages_found = 0
    for i, page in enumerate(doc):
        text  = page.get_text("text")
        lines = text.splitlines()
        if lines and "MOVEMENT SUMMARY" in lines[0]:
            new_doc = fitz.open()
            new_doc.insert_pdf(doc, from_page=i, to_page=i)
            out_path = os.path.join(OUTPUT_FOLDER, f"Sidra_JN_{count + 1}.pdf")
            new_doc.save(out_path)
            count       += 1
            pages_found += 1
    print(f"   📄 {filename}  →  {pages_found} MOVEMENT SUMMARY page(s) found")

print(f"\n✅ Total pages extracted: {count}")

## Step 5 – Parse Each Page and Extract Summary Data

For every extracted page this step:
- Reads the **Scenario** (e.g. Base AM) and **Location** (e.g. Jn 1) from the header
- Reads each **Approach** row (SouthEast, NorthWest, etc.) and its delay
- Reads the **All Vehicles** row for the overall intersection delay
- Calculates LOS from delay for both overall and worst approach
- Identifies the **Worst Approach** as the one with the highest delay

The final table is sorted by **Scenario** first, then by **junction number** (1, 2, 3 … 10, 11 …) so the order is always correct regardless of suffix or prefix in the junction name.

In [ ]:
import re

# ── Natural sort key: extracts the leading integer from a filename ────────
# e.g. "Sidra_JN_10.pdf" → 10  so files sort as 1, 2, 3 … 10, 11 …
def natural_sort_key(filename):
    numbers = re.findall(r"\d+", filename)
    return [int(n) for n in numbers]


# ── Junction number extractor for sorting the output DataFrame ───────────
# Works on any Location string: "Jn 1", "Jn 10 -RIRO", "Jn 23a" → 1, 10, 23
def junction_sort_key(location: str) -> int:
    m = re.search(r"(\d+)", str(location))
    return int(m.group(1)) if m else 0


# ── Regex to detect approach direction headers  e.g. "SouthEast: RoadName"
APPROACH_RE = re.compile(
    r"^(North(?:East|West)?|South(?:East|West)?|East|West|North|South)\s*:\s*",
    re.IGNORECASE,
)

results = []

# ── Use natural sort so Sidra_JN_2.pdf comes before Sidra_JN_10.pdf ──────
pdf_files = sorted(
    [f for f in os.listdir(OUTPUT_FOLDER) if f.endswith(".pdf")],
    key=natural_sort_key
)

for pdf_file in pdf_files:
    with pdfplumber.open(os.path.join(OUTPUT_FOLDER, pdf_file)) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text:
                continue

            lines = [ln.strip() for ln in text.splitlines()]

            # ── 1. Extract Scenario and Location from the page header ─────
            scenario = tmc_number = None
            for line in lines:
                m = re.search(
                    r"Site:\s*\S+\s*\[(.+?)\s*\(Site Folder:\s*(.+?)\)\]", line
                )
                if m:
                    tmc_number = m.group(1).strip()   # e.g. "Jn 1"
                    scenario   = m.group(2).strip()   # e.g. "Base AM"
                    break
            if not scenario:
                continue

            # ── 2. Walk through lines and collect delays ──────────────────
            approach_delays  = {}    # {direction: delay}
            overall_delay    = None
            current_approach = None

            for line in lines:

                # Approach direction header  (e.g. "SouthEast: RoadName")
                am = APPROACH_RE.match(line)
                if am:
                    current_approach = am.group(1).strip()
                    continue

                # Approach summary row
                # Format: Approach | flow | HV% | flow | HV% | v/c | delay | ...
                if line.startswith("Approach") and current_approach:
                    nums = []
                    for p in line.split()[1:]:
                        try:
                            nums.append(float(p))
                        except ValueError:
                            pass
                    if len(nums) >= 6:
                        dv = nums[5]   # 6th numeric value = Aver. Delay
                        if (current_approach not in approach_delays
                                or dv > approach_delays[current_approach]):
                            approach_delays[current_approach] = dv
                    continue

                # All Vehicles row  →  overall intersection delay
                if line.startswith("All Vehicles"):
                    nums = []
                    for p in line.split()[1:]:
                        try:
                            nums.append(float(p))
                        except ValueError:
                            pass
                    if len(nums) >= 6:
                        overall_delay = nums[5]
                    continue

            if overall_delay is None:
                continue

            # ── 3. Identify the worst approach (highest delay) ────────────
            worst_approach       = None
            worst_approach_delay = None
            worst_approach_los   = None

            if approach_delays:
                worst_approach       = max(approach_delays, key=approach_delays.get)
                worst_approach_delay = approach_delays[worst_approach]
                worst_approach_los   = delay_to_los(worst_approach_delay)

            results.append({
                "Scenario":                         scenario,
                "Location":                         tmc_number,
                "Overall Intersection Delay (sec)": overall_delay,
                "Overall LOS":                      delay_to_los(overall_delay),
                "Worst Approach":                   worst_approach,
                "Worst Approach Delay (sec)":       worst_approach_delay,
                "Worst Approach LOS":               worst_approach_los,
            })

# ── 4. Build DataFrame and sort: Scenario order preserved, then Jn number ─
Output = pd.DataFrame(results)

# Assign a numeric sort key for the junction number (ignores prefix/suffix)
Output["_jn_num"] = Output["Location"].apply(junction_sort_key)

# Sort: keep Scenario in the order it was encountered, sort junctions numerically
scenario_order = Output["Scenario"].unique().tolist()
Output["_scen_order"] = Output["Scenario"].apply(lambda s: scenario_order.index(s))
Output = (
    Output
    .sort_values(["_scen_order", "_jn_num"])
    .drop(columns=["_jn_num", "_scen_order"])
    .reset_index(drop=True)
)

print(f"✅ Parsed {len(Output)} junction(s) successfully.")
print(f"   Scenarios : {Output['Scenario'].unique().tolist()}")
print(f"   Order check (first 5): {Output['Location'].head().tolist()}\n")
Output

## Step 6 – Export Results to Excel

Saves the summary table to **Sidra_Summary_Report.xlsx** and downloads it automatically.

In [ ]:
OUTPUT_FILE = "Sidra_Summary_Report.xlsx"

Output.to_excel(OUTPUT_FILE, index=False)
print(f"✅ Report saved as '{OUTPUT_FILE}'")
print(f"   Rows    : {len(Output)}")
print(f"   Columns : {list(Output.columns)}")

# Automatically download the file
files.download(OUTPUT_FILE)
print("\n⬇️  Download started.")

---
## ✅ Done!

Your **Sidra_Summary_Report.xlsx** has been downloaded.

### Troubleshooting
| Issue | Fix |
|-------|-----|
| Empty output | Make sure the PDF contains pages with a `MOVEMENT SUMMARY` header |
| Missing junctions | Check that every page has the `Site: X [Jn X (Site Folder: ...)]` header line |
| Wrong delay values | Confirm the PDF was exported directly from SIDRA (not a scanned image) |